# เตรียมข้อมูลแบบเก็บทุกชุด 5 epoch ที่ติดกัน (ไม่ซ้ำกัน)

## Section 1 Import และ Path Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install mne -q

import json
import numpy as np
import pandas as pd
import mne
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/SleepStageClassificationProject"
DATA_DIR = Path(f"{PROJECT_DIR}/dataset/HMC")
OLD_DIR = Path(f"{PROJECT_DIR}/dataset/preprocess_microvolt")
OUT_DIR = Path(f"{PROJECT_DIR}/dataset/preprocess_microvolt_multi5")

for name in ["train", "val", "test"]:
    (OUT_DIR / name).mkdir(parents=True, exist_ok=True)

FS = 256
EPOCH_LEN = 30 * FS
N_CONSEC = 5
STAGES = ["W", "N1", "N2", "N3", "R"]

CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "chin"]
CHANNEL_NAMES = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin EMG"]

MAX_GROUPS_PER_STAGE = None

## Section 2 ใช้ split เดิม

In [ ]:
old_split_path = OLD_DIR / "split_subjects.json"
assert old_split_path.exists(), f"ไม่พบ split เดิม: {old_split_path}"

with open(old_split_path) as f:
    split_info = json.load(f)

with open(OUT_DIR / "split_subjects.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Train:", split_info["n_train"], "| Val:", split_info["n_val"], "| Test:", split_info["n_test"])

## Section 3 ฟังก์ชัน

In [ ]:
def get_labels(subject, n_epochs):
    ann = mne.read_annotations(DATA_DIR / f"{subject}_sleepscoring.edf")
    labels = ["?"] * n_epochs
    onsets = [None] * n_epochs

    for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
        stage = desc.split()[-1]
        if stage not in STAGES:
            continue
        first = int(round(onset / 30))
        last = int(round((onset + duration) / 30))
        for e in range(first, last):
            if e < n_epochs:
                labels[e] = stage
                onsets[e] = onset + (e - first) * 30
    return labels, onsets


def pick_channels(raw):
    picks = []
    for key in CHANNELS:
        found = [i for i, n in enumerate(raw.ch_names) if key.lower() in n.lower()]
        if len(found) != 1:
            return None
        picks.append(found[0])
    return picks


def find_all_runs(labels, stage):
    starts = []
    i = 0
    while i + N_CONSEC <= len(labels):
        if all(labels[i + k] == stage for k in range(N_CONSEC)):
            starts.append(i)
            i += N_CONSEC
        else:
            i += 1
    return starts

def choose_starts(starts):
    if MAX_GROUPS_PER_STAGE is None or len(starts) <= MAX_GROUPS_PER_STAGE:
        return starts
    pick = np.linspace(0, len(starts) - 1, MAX_GROUPS_PER_STAGE, dtype=int)
    return [starts[i] for i in pick]

## Section 4 นับจำนวนชุดก่อนบันทึก (ยังไม่เขียนไฟล์)

อ่านเฉพาะป้ายระยะ ไม่อ่านสัญญาณ

In [ ]:
count_rows = []

for split in ["train", "val", "test"]:
    for subject in split_info[f"{split}_subjects"]:
        raw = mne.io.read_raw_edf(DATA_DIR / f"{subject}.edf", preload=False)
        row = {"split": split, "subject": subject}

        if pick_channels(raw) is None:
            for s in STAGES:
                row[s] = 0
        else:
            labels, onsets = get_labels(subject, raw.n_times // EPOCH_LEN)
            for s in STAGES:
                row[s] = len(choose_starts(find_all_runs(labels, s)))
        count_rows.append(row)

counts = pd.DataFrame(count_rows)
counts["complete"] = (counts[STAGES] > 0).all(axis=1)
kept = counts[counts["complete"]].copy()
kept["total"] = kept[STAGES].sum(axis=1)
dropped = counts[~counts["complete"]]

print("Subject ที่เก็บ (มีครบ 5 ระยะ):", kept.groupby("split").size().to_dict())
print("Subject ที่ตัดออก (ขาดบางระยะ):", len(dropped))
if len(dropped) > 0:
    print(dropped[["split", "subject"] + STAGES].to_string(index=False))

print("\nจำนวนชุดรวมของ subject ที่เก็บ ตาม split และระยะ")
print(kept.groupby("split")[STAGES + ["total"]].sum())

print("\nต่อ subject (ค่ากลาง / ต่ำสุด / สูงสุด)")
print(kept.groupby("split")["total"].agg(["median", "min", "max"]))

bytes_saved = N_CONSEC * EPOCH_LEN * len(CHANNELS) * 4
bytes_train_ram = N_CONSEC * 3000 * len(CHANNELS) * 4
total_groups = int(kept["total"].sum())
train_groups = int(kept.loc[kept["split"] == "train", "total"].sum())

print(f"\nขนาดไฟล์โดยประมาณทั้งหมด: {total_groups * bytes_saved / 1e9:.1f} GB")
print(f"RAM ที่ชุด train ใช้ตอนเทรน (100 Hz): {train_groups * bytes_train_ram / 1e9:.1f} GB (ต้องมี RAM ว่างประมาณ 2 เท่าของค่านี้)")

## Section 5 สร้างไฟล์

In [ ]:
def process_subject(subject, split):
    raw = mne.io.read_raw_edf(DATA_DIR / f"{subject}.edf", preload=False)

    if raw.info["sfreq"] != FS:
        raise ValueError(f"{subject}: sampling rate = {raw.info['sfreq']}")

    picks = pick_channels(raw)
    if picks is None:
        return {"subject": subject, "split": split, "n_groups": 0, "saved": False, "missing_stages": "missing channels"}

    labels, onsets = get_labels(subject, raw.n_times // EPOCH_LEN)

    all_starts = {s: choose_starts(find_all_runs(labels, s)) for s in STAGES}
    missing = [s for s in STAGES if len(all_starts[s]) == 0]

    if missing:
        (OUT_DIR / split / f"{subject}.npz").unlink(missing_ok=True)
        return {"subject": subject, "split": split, "n_groups": 0, "saved": False, "missing_stages": ",".join(missing)}

    X, y, stage, epoch_indices = [], [], [], []
    start_samples, end_samples = [], []

    for s in STAGES:
        for first in all_starts[s]:
            idx = list(range(first, first + N_CONSEC))
            data_uv = raw.get_data(
                picks=picks,
                start=first * EPOCH_LEN,
                stop=(first + N_CONSEC) * EPOCH_LEN,
                units="uV"
            )
            x = data_uv.T.reshape(N_CONSEC, EPOCH_LEN, len(picks))

            X.append(x.astype(np.float32))
            y.append([labels[e] for e in idx])
            stage.append(s)
            epoch_indices.append(idx)
            start_samples.append([e * EPOCH_LEN for e in idx])
            end_samples.append([(e + 1) * EPOCH_LEN - 1 for e in idx])

    np.savez(
        OUT_DIR / split / f"{subject}.npz",
        X=np.array(X, dtype=np.float32),
        y=np.array(y),
        subject_id=subject,
        stage=np.array(stage),
        epoch_indices=np.array(epoch_indices),
        start_samples=np.array(start_samples),
        end_samples=np.array(end_samples),
        n_valid=np.full(len(X), N_CONSEC)
    )

    return {
        "subject": subject,
        "split": split,
        "n_groups": len(X),
        "saved": True,
        "missing_stages": ",".join(missing)
    }


rows = []
for split in ["train", "val", "test"]:
    for subject in split_info[f"{split}_subjects"]:
        row = process_subject(subject, split)
        rows.append(row)
        print(split, subject, "groups:", row["n_groups"], "missing:", row["missing_stages"])

summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "preprocessing_summary.csv", index=False)

## Section 6 ตรวจสอบความถูกต้อง

ตรวจว่าทุกชุดเป็น 5 epoch ติดกัน ระยะเดียวกัน ไม่ซ้อนกัน ตรงกับ Ground Truth และสัญญาณตรงกับไฟล์ EDF และชุดแรกของแต่ละระยะต้องตรงกับข้อมูลเดิม

In [ ]:
ver_rows = []

for _, r in summary[summary["saved"]].iterrows():
    subject, split = r["subject"], r["split"]

    data = np.load(OUT_DIR / split / f"{subject}.npz", allow_pickle=True)
    raw = mne.io.read_raw_edf(DATA_DIR / f"{subject}.edf", preload=False)
    picks = pick_channels(raw)
    labels, onsets = get_labels(subject, raw.n_times // EPOCH_LEN)

    stages = [str(s) for s in data["stage"]]
    X_all = data["X"]
    ep_all = data["epoch_indices"]
    st_all = data["start_samples"]
    en_all = data["end_samples"]
    y_all = data["y"]

    assert X_all.dtype == np.float32, f"{subject}: X ไม่ใช่ float32"
    assert X_all.shape[1:] == (N_CONSEC, EPOCH_LEN, len(picks)), f"{subject}: shape ของ X ผิด {X_all.shape}"
    assert np.all(data["n_valid"] == N_CONSEC), f"{subject}: n_valid ไม่ใช่ {N_CONSEC}"

    for s in STAGES:
        stored = [int(ep_all[g][0]) for g in range(len(stages)) if stages[g] == s]
        expected = choose_starts(find_all_runs(labels, s))
        assert stored == expected, f"{subject} / {s}: ชุดที่เก็บไม่ตรงกับที่คาด"
        assert all(b - a >= N_CONSEC for a, b in zip(stored, stored[1:])), f"{subject} / {s}: มีชุดซ้อนกัน"

    for g in range(len(stages)):
        stage = stages[g]
        ep = np.asarray(ep_all[g], dtype=int)
        st = np.asarray(st_all[g], dtype=int)
        en = np.asarray(en_all[g], dtype=int)
        y = np.asarray(y_all[g])
        where = f"{subject} / {stage} / epochs {list(ep)}"

        assert np.all(np.diff(ep) == 1), f"{where}: epoch ไม่ต่อเนื่อง"
        assert all(v == stage for v in y), f"{where}: y ไม่ใช่ stage เดียวกัน"

        for k in range(N_CONSEC):
            e = int(ep[k])
            w = f"{subject} / {stage} / epoch {e}"

            assert labels[e] == stage, f"{w}: Ground Truth คือ {labels[e]}"
            assert st[k] == e * EPOCH_LEN, f"{w}: start sample ผิด"
            assert en[k] == (e + 1) * EPOCH_LEN - 1, f"{w}: end sample ผิด"
            assert abs(onsets[e] - st[k] / FS) < 1e-6, f"{w}: onset ไม่ตรงกับ sample"

            sig = raw.get_data(picks=picks, start=int(st[k]), stop=int(en[k]) + 1, units="uV").T
            assert np.array_equal(sig.astype(np.float32), X_all[g][k]), f"{w}: สัญญาณไม่ตรงกับ EDF"

        ver_rows.append({"Subject": subject, "Stage": stage, "First Epoch": int(ep[0])})

verification = pd.DataFrame(ver_rows)
verification.to_csv(OUT_DIR / "verification.csv", index=False)
print("Verification ผ่านทั้งหมด:", len(verification), "ชุด")

## Section 7 สรุปผล

In [ ]:
print("Subject ทั้งหมดใน split:", len(summary))
print("Subject ที่เก็บ (ครบ 5 ระยะ):", int(summary["saved"].sum()))
print("Subject ที่ตัดออก:", int((~summary["saved"]).sum()))

for split in ["train", "val", "test"]:
    d = summary[summary["split"] == split]
    v = verification[verification["Subject"].isin(d["subject"])]

    print(f"\n[{split}]")
    print("Subjects ที่เก็บ:", int(d["saved"].sum()), "| ตัดออก:", int((~d["saved"]).sum()))
    for _, r in d[~d["saved"]].iterrows():
        print("  ", r["subject"], "ขาด:", r["missing_stages"])
    print("จำนวนชุดทั้งหมด:", len(v))
    print(v["Stage"].value_counts().reindex(STAGES).fillna(0).astype(int).to_string())

print("\nเทียบกับข้อมูลเดิม")
for split in ["train", "val", "test"]:
    old_ok = set()
    for f in sorted((OLD_DIR / split).glob("SN*.npz")):
        with np.load(f, allow_pickle=True) as d:
            full = {str(s) for s, n in zip(d["stage"], d["n_valid"]) if int(n) == N_CONSEC}
        if full == set(STAGES):
            old_ok.add(f.stem)
    new_ok = {f.stem for f in (OUT_DIR / split).glob("SN*.npz")}
    print(split, "| เดิม:", len(old_ok), "| ใหม่:", len(new_ok), "| ตรงกัน:", old_ok == new_ok)
    assert old_ok == new_ok, f"{split}: subject ไม่ตรงกับข้อมูลเดิม"

print("\nแต่ละชุด = 5 epoch ติดกัน ระยะเดียวกัน ไม่ซ้อนกัน")
print("Signal unit: µV | float32 | 256 Hz | Channels:", CHANNEL_NAMES)
print("Output directory:", OUT_DIR)

In [ ]:
kept["stored_epochs"] = kept["total"] * N_CONSEC

print("\nจำนวน sequences และ epochs ที่จะบันทึก")
print(kept.groupby("split")[["total", "stored_epochs"]].sum()
      .rename(columns={"total": "sequences", "stored_epochs": "epochs"}))

print("\nจำนวน sequences ต่อ subject")
display(kept[["split", "subject", "W", "N1", "N2", "N3", "R",
              "total", "stored_epochs"]].sort_values(["split", "subject"]))

In [ ]:
for split in ["train", "val", "test"]:
    for f in sorted((OUT_DIR / split).glob("SN*.npz")):
        with np.load(f, allow_pickle=True) as d:
            print(split, f.stem,
                  "sequences:", len(d["X"]),
                  "epochs:", len(d["X"]) * N_CONSEC)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

OLD_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject/dataset/preprocess_microvolt")
STAGES = ["W", "N1", "N2", "N3", "R"]

rows = []

for split in ["train", "val", "test"]:
    for f in sorted((OLD_DIR / split).glob("SN*.npz")):
        with np.load(f, allow_pickle=True) as d:
            stages = d["stage"].astype(str)
            n_valid = d["n_valid"].astype(int)

            row = {
                "split": split,
                "subject": f.stem,
                "sequences": len(stages),
                "epochs": int(n_valid.sum()),
            }

            for s in STAGES:
                row[s] = int(np.sum(stages == s))

            row["has_all_5_stages"] = all(row[s] > 0 for s in STAGES)
            rows.append(row)

old_counts = pd.DataFrame(rows)

print("สรุปจำนวนข้อมูล preprocessing เก่า")
display(
    old_counts.groupby("split").agg(
        subjects=("subject", "nunique"),
        sequences=("sequences", "sum"),
        epochs=("epochs", "sum"),
        complete_subjects=("has_all_5_stages", "sum"),
    )
)

print("\nจำนวน sequences แยกตามระยะ")
display(old_counts.groupby("split")[STAGES].sum())

print("\nจำนวน sequences ต่อ subject")
display(old_counts.sort_values(["split", "subject"]))